[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdn-cs502k-symbolic-ai/practicals/blob/main/day03/tutorial3-logic-solutions.ipynb)


# CS502K: Symbolic Artificial Intelligence

## Tutorial 03: Logic and Inference - Solutions

#### Prof. Felipe Meneguzzi


This notebook is the coded half of Tutorial 3 and follows Lectures 5 and 6.
Work through the two paper tutorials first.
Every algorithm here is one you traced by hand there, and the tests below check your code against the answers you reached.

The propositional half covers questions 1 to 5 of Logic 1, and the first-order half covers questions 2, 4 and 8 of Logic 2.

Three of the algorithms here are described in the **bonus deck** rather than in the lectures: `to_cnf`, `pl_resolution` and the occurs check are all in *Bonus: Normal Forms and Resolution*.
Read it first. None of it is examined; it is here for depth, and this notebook is where it pays off.

This is the worked version, released at the end of the week.
Read it after you have tried the stubs yourself; a solution you have not struggled against teaches very little.


In [ ]:
try:
    import google.colab
    print("We are in Google colab, we need to clone the repo")
    !git clone https://github.com/abdn-cs502k-symbolic-ai/practicals.git
    %cd practicals/day03
except:
    print("Not in colab")

# This notebook needs nothing else: everything it uses is defined below, so it
# also runs on its own if you downloaded it from MyAberdeen.

## Representing sentences

We need no library for this.
A propositional sentence is either a string, which is a proposition symbol, or a tuple whose first element names the connective:

```
"P"                     the symbol P
("~", "P")              ~P
("&", "P", "Q")         (P & Q)
("|", "P", "Q")         (P | Q)
("=>", "P", "Q")        (P => Q)
("<=>", "P", "Q")       (P <=> Q)
```

The helper constructors below build these tuples, and `And` and `Or` take any number of arguments so that `And(R1, R2, R3, R4, R5)` reads the way the knowledge base does.
Everything that follows works on this representation, so you can print any sentence and see exactly what your code built.


In [ ]:
import itertools
from functools import reduce

def Not(a): return ("~", a)
def And(*xs): return reduce(lambda a, b: ("&", a, b), xs)
def Or(*xs):  return reduce(lambda a, b: ("|", a, b), xs)
def Implies(a, b): return ("=>", a, b)
def Iff(a, b): return ("<=>", a, b)

def is_symbol(s): return isinstance(s, str)

def pretty(s):
    if is_symbol(s): return s
    op = s[0]
    if op == "~": return "~" + pretty(s[1])
    return "(" + pretty(s[1]) + " " + op + " " + pretty(s[2]) + ")"

def symbols(s):
    if is_symbol(s): return {s}
    if s[0] == "~": return symbols(s[1])
    return symbols(s[1]) | symbols(s[2])

## The test harness

`check` compares one value against the expected one and keeps going either way.
`report` prints the totals at the end.


In [ ]:
PASS, FAIL = 0, 0

def check(label, got, want):
    """Report one assertion without stopping the notebook on the first failure."""
    global PASS, FAIL
    if got == want:
        PASS += 1
        print("PASS  " + label)
    else:
        FAIL += 1
        print("FAIL  " + label + "\n      got  " + repr(got) + "\n      want " + repr(want))

def report():
    print("\n%d PASS, %d FAIL" % (PASS, FAIL))

## 1. The interpretation function

Question 1(f) of Logic 1 asks you to compute $I(m, ((P \Rightarrow \lnot Q) \land R))$ by hand for one model.
`pl_true(model, sentence)` is that function, and every algorithm below is built on it.


In [ ]:
def pl_true(model, s):
    if is_symbol(s): return model[s]
    op = s[0]
    if op == "~":   return not pl_true(model, s[1])
    a = pl_true(model, s[1]); b = pl_true(model, s[2])
    if op == "&":   return a and b
    if op == "|":   return a or b
    if op == "=>":  return (not a) or b
    if op == "<=>": return a == b
    raise ValueError(op)

In [ ]:
m = {"P": False, "Q": False, "R": True}

check("1(f): I(m, ((P => ~Q) & R))", pl_true(m, And(Implies("P", Not("Q")), "R")), True)
check("~P under P=False", pl_true({"P": False}, Not("P")), True)
check("(P => Q) with P false", pl_true({"P": False, "Q": False}, Implies("P", "Q")), True)
check("(P <=> Q) with P true, Q false", pl_true({"P": True, "Q": False}, Iff("P", "Q")), False)
report()

## 2. Model enumeration, and the Wumpus knowledge base

`tt_entails` is the algorithm of question 1(d), under the lecture's names.
Question 2 then asks you to work out which of eight models satisfy the Wumpus knowledge base, and what the agent may conclude.
Your code should agree with the three models you found on paper.


In [ ]:
def tt_entails(kb, alpha):
    syms = sorted(symbols(kb) | symbols(alpha))
    return tt_check_all(kb, alpha, syms, {})

def tt_check_all(kb, alpha, syms, model):
    if not syms:
        if pl_true(model, kb): return pl_true(model, alpha)
        return True
    p, rest = syms[0], syms[1:]
    return (tt_check_all(kb, alpha, rest, {**model, p: True})
            and tt_check_all(kb, alpha, rest, {**model, p: False}))

def models_of(s):
    syms = sorted(symbols(s))
    out = []
    for vals in itertools.product([False, True], repeat=len(syms)):
        m = dict(zip(syms, vals))
        if pl_true(m, s): out.append(m)
    return out

In [ ]:
# Question 1(d): an inconsistent KB entails everything.
kb_inconsistent = And(And("P", Not("P")), Not("Q"))
check("1(d): inconsistent KB entails Q", tt_entails(kb_inconsistent, "Q"), True)
check("1(d): and entails ~Q as well", tt_entails(kb_inconsistent, Not("Q")), True)

# Question 2: the Wumpus KB after visiting [1,1] and [2,1].
R1 = Not("P11")
R2 = Not("B11")
R3 = "B21"
R4 = Iff("B11", Or("P12", "P21"))
R5 = Iff("B21", Or("P11", "P22", "P31"))
wumpus = And(R1, R2, R3, R4, R5)

check("2(b): three models satisfy KB", len(models_of(wumpus)), 3)
check("2(c): KB entails ~P12", tt_entails(wumpus, Not("P12")), True)
check("2(c): KB does not entail ~P22", tt_entails(wumpus, Not("P22")), False)
check("2(c): KB does not entail P22 either", tt_entails(wumpus, "P22"), False)
report()

## 3. Conjunctive normal form

Question 3 asks for the four passes in pseudo-code.
Here you write them.
The test does not compare your output against one expected string, because a sentence has many correct CNFs that differ only in the order of clauses.
It compares truth tables instead, which is the definition of logical equivalence and the only check that is fair to your implementation.


In [ ]:
def eliminate_iff(s):
    if is_symbol(s): return s
    if s[0] == "~": return Not(eliminate_iff(s[1]))
    a, b = eliminate_iff(s[1]), eliminate_iff(s[2])
    if s[0] == "<=>": return And(Implies(a, b), Implies(b, a))
    return (s[0], a, b)

def eliminate_implies(s):
    if is_symbol(s): return s
    if s[0] == "~": return Not(eliminate_implies(s[1]))
    a, b = eliminate_implies(s[1]), eliminate_implies(s[2])
    if s[0] == "=>": return Or(Not(a), b)
    return (s[0], a, b)

def move_not_inwards(s):
    if is_symbol(s): return s
    if s[0] == "~":
        a = s[1]
        if is_symbol(a): return s
        if a[0] == "~": return move_not_inwards(a[1])
        if a[0] == "&": return Or(move_not_inwards(Not(a[1])), move_not_inwards(Not(a[2])))
        if a[0] == "|": return And(move_not_inwards(Not(a[1])), move_not_inwards(Not(a[2])))
        raise ValueError(a[0])
    return (s[0], move_not_inwards(s[1]), move_not_inwards(s[2]))

def distribute_or_over_and(s):
    if is_symbol(s) or s[0] == "~": return s
    if s[0] == "&":
        return And(distribute_or_over_and(s[1]), distribute_or_over_and(s[2]))
    a = distribute_or_over_and(s[1]); b = distribute_or_over_and(s[2])
    if not is_symbol(a) and a[0] == "&":
        return distribute_or_over_and(And(Or(a[1], b), Or(a[2], b)))
    if not is_symbol(b) and b[0] == "&":
        return distribute_or_over_and(And(Or(a, b[1]), Or(a, b[2])))
    return Or(a, b)

def to_cnf(s):
    return distribute_or_over_and(move_not_inwards(eliminate_implies(eliminate_iff(s))))

In [ ]:
def equivalent(a, b):
    """True when a and b have the same truth value in every model."""
    syms = sorted(symbols(a) | symbols(b))
    for vals in itertools.product([False, True], repeat=len(syms)):
        m = dict(zip(syms, vals))
        if pl_true(m, a) != pl_true(m, b):
            return False
    return True


def is_cnf(s):
    """True when s is a conjunction of disjunctions of literals."""
    def literal(x):
        return is_symbol(x) or (x[0] == "~" and is_symbol(x[1]))

    def clause(x):
        return literal(x) or (x[0] == "|" and clause(x[1]) and clause(x[2]))

    return clause(s) or (not is_symbol(s) and s[0] == "&" and is_cnf(s[1]) and is_cnf(s[2]))


examples = [
    ("A <=> (B | C)", Iff("A", Or("B", "C"))),
    ("(P & Q) => R", Implies(And("P", "Q"), "R")),
    ("~(P & (Q | ~R))", Not(And("P", Or("Q", Not("R"))))),
    ("contraposition", Iff(Implies("P", "Q"), Implies(Not("Q"), Not("P")))),
    ("the Wumpus KB", wumpus),
]
for label, s in examples:
    check("3: to_cnf preserves meaning, " + label, equivalent(s, to_cnf(s)), True)
    check("3: to_cnf output is in CNF, " + label, is_cnf(to_cnf(s)), True)
report()

## 4. Resolution

Question 4 asks for a refutation of $KB \land \lnot \alpha$ where $KB = (A \Leftrightarrow (B \lor C)) \land \lnot B \land \lnot C$ and $\alpha = \lnot A$, and then asks what changes when $\lnot C$ is dropped.
Your `pl_resolution` should reproduce both verdicts.

The last test is the one worth caring about.
Resolution and model enumeration are completely different algorithms, so where they disagree one of them is wrong, and in practice the bug is a missing case in `to_cnf`.


In [ ]:
def conjuncts(s):
    if not is_symbol(s) and s[0] == "&":
        return conjuncts(s[1]) + conjuncts(s[2])
    return [s]

def disjuncts(s):
    if not is_symbol(s) and s[0] == "|":
        return disjuncts(s[1]) + disjuncts(s[2])
    return [s]

def clauses(s):
    return [frozenset(disjuncts(c)) for c in conjuncts(to_cnf(s))]

def negate_literal(lit):
    return lit[1] if (not is_symbol(lit) and lit[0] == "~") else Not(lit)

def resolve(ci, cj):
    out = []
    for lit in ci:
        if negate_literal(lit) in cj:
            out.append((ci - {lit}) | (cj - {negate_literal(lit)}))
    return out

def pl_resolution(kb, alpha):
    cs = set(clauses(And(kb, Not(alpha))))
    new = set()
    while True:
        pairs = [(ci, cj) for i, ci in enumerate(cs) for cj in list(cs)[i + 1:]]
        for ci, cj in pairs:
            for rc in resolve(ci, cj):
                if not rc: return True
                new.add(rc)
        if new <= cs: return False
        cs |= new

In [ ]:
kb_q4 = And(Iff("A", Or("B", "C")), And(Not("B"), Not("C")))
check("4(b): refutation succeeds", pl_resolution(kb_q4, Not("A")), True)

kb_q4c = And(Iff("A", Or("B", "C")), Not("B"))
check("4(c): without ~C there is no refutation", pl_resolution(kb_q4c, Not("A")), False)
check("4(c): and the model A,~B,C satisfies the KB",
      pl_true({"A": True, "B": False, "C": True}, kb_q4c), True)

check("2(c) again, by resolution", pl_resolution(wumpus, Not("P12")), True)

# Resolution and enumeration must agree on everything.
queries = [("A", "B"), (kb_q4, "A"), (kb_q4, Not("A")), (wumpus, Not("P12")),
           (wumpus, Not("P22")), (And("P", Implies("P", "Q")), "Q"),
           (Implies("P", "Q"), Implies(Not("Q"), Not("P")))]
agree = all(tt_entails(k, a) == pl_resolution(k, a) for k, a in queries)
check("4: resolution agrees with model enumeration", agree, True)
report()

## 5. Unification

This is the first-order half, and it starts with question 2 of Logic 2.
Terms follow the same convention as sentences: a tuple is a compound term whose first element is the function or predicate name, a lowercase string is a variable, and anything else is a constant.

Write `unify` without the occurs check first.
The cell after it has one test that should fail while the check is missing, which is the point of question 4(b).


In [ ]:
FAILURE = "FAILURE"

def is_var(x): return isinstance(x, str) and x[0].islower()

def unify(x, y, subst=None, occurs_check=True):
    if subst is None: subst = {}
    if subst is FAILURE: return FAILURE
    if x == y: return subst
    if is_var(x): return unify_var(x, y, subst, occurs_check)
    if is_var(y): return unify_var(y, x, subst, occurs_check)
    if isinstance(x, tuple) and isinstance(y, tuple):
        if len(x) != len(y) or x[0] != y[0]: return FAILURE
        for a, b in zip(x[1:], y[1:]):
            subst = unify(a, b, subst, occurs_check)
            if subst is FAILURE: return FAILURE
        return subst
    return FAILURE

def unify_var(v, x, subst, occurs_check):
    if v in subst: return unify(subst[v], x, subst, occurs_check)
    if isinstance(x, str) and x in subst: return unify(v, subst[x], subst, occurs_check)
    if occurs_check and occurs(v, x, subst): return FAILURE
    return {**subst, v: x}

def occurs(v, x, subst):
    if v == x: return True
    if isinstance(x, str) and x in subst: return occurs(v, subst[x], subst)
    if isinstance(x, tuple): return any(occurs(v, a, subst) for a in x[1:])
    return False

def subst_apply(subst, x):
    if isinstance(x, str):
        return subst_apply(subst, subst[x]) if x in subst else x
    return (x[0],) + tuple(subst_apply(subst, a) for a in x[1:])

In [ ]:
def T(name, *args):
    """A compound term: T("Father", "y") is Father(y)."""
    return (name,) + args


def resolved(subst):
    """A substitution with every chain of bindings followed to the end."""
    if subst is FAILURE:
        return FAILURE
    return {v: subst_apply(subst, v) for v in subst}


check("2(a): P(z,y,x) with P(B,C,C)",
      resolved(unify(T("P", "z", "y", "x"), T("P", "B", "C", "C"))),
      {"z": "B", "y": "C", "x": "C"})
check("2(b): Q(G(z,z),x) with Q(x,G(A,B)) has no unifier",
      unify(T("Q", T("G", "z", "z"), "x"), T("Q", "x", T("G", "A", "B"))), FAILURE)
check("2(c): Older(Father(y),Paul) with Older(Father(z),z)",
      resolved(unify(T("Older", T("Father", "y"), "Paul"), T("Older", T("Father", "z"), "z"))),
      {"y": "Paul", "z": "Paul"})
check("2(d): Knows(Father(x),x) with Knows(z,z) fails the occurs check",
      unify(T("Knows", T("Father", "x"), "x"), T("Knows", "z", "z")), FAILURE)

# Question 4(b). Skolemising forall x exists y P(x,y) gives P(x,F(x)), and the
# negated goal exists z P(z,z) gives ~P(z,z). The two must NOT unify.
check("4(b): P(x,F(x)) does not unify with P(z,z)",
      unify(T("P", "x", T("F", "x")), T("P", "z", "z")), FAILURE)
check("4(b): without the occurs check it does, which is the bug",
      unify(T("P", "x", T("F", "x")), T("P", "z", "z"), occurs_check=False) is not FAILURE, True)
report()

## 6. Backward chaining with unification

Question 3 of Logic 2 asks you to write the bird knowledge base as definite clauses.
Here it is, as a list of `(premises, conclusion)` pairs where a fact has no premises.
`fol_bc_ask` should prove a goal by unifying it with the conclusion of a rule and then proving that rule's premises, which is the algorithm of Lecture 6 lifted by the unifier you just wrote.

Two details matter.
Variables in a rule must be renamed on every use, or two applications of the same rule capture each other's variables.
And the recursion needs a depth limit, because `Offspring` and `Parent` are defined in terms of each other and will otherwise chase each other forever.


In [ ]:
def standardise(rule, n):
    def rn(t):
        if isinstance(t, str): return t + "_" + str(n) if is_var(t) else t
        return (t[0],) + tuple(rn(a) for a in t[1:])
    prem, concl = rule
    return [rn(p) for p in prem], rn(concl)

def fol_bc_ask(kb, goal, subst=None, depth=0, counter=None):
    """Yield substitutions proving goal. kb is a list of (premises, conclusion)."""
    if subst is None: subst = {}
    if counter is None: counter = itertools.count()
    if depth > 12: return
    for rule in kb:
        prem, concl = standardise(rule, next(counter))
        s = unify(concl, subst_apply(subst, goal), dict(subst))
        if s is FAILURE: continue
        yield from bc_and(kb, prem, s, depth + 1, counter)

def bc_and(kb, goals, subst, depth, counter):
    if not goals:
        yield subst
        return
    first, rest = goals[0], goals[1:]
    for s in fol_bc_ask(kb, subst_apply(subst, first), subst, depth, counter):
        yield from bc_and(kb, rest, s, depth, counter)

In [ ]:
bird_kb = [
    ([T("Magpie", "x")], T("Bird", "x")),
    ([T("Pheasant", "x")], T("Bird", "x")),
    ([T("Pigeon", "x")], T("Bird", "x")),
    ([T("Offspring", "x", "y"), T("Bird", "y")], T("Bird", "x")),
    ([], T("Magpie", "Tam")),
    ([], T("Parent", "Tam", "Doug")),
    ([T("Offspring", "x", "y")], T("Parent", "y", "x")),
    ([T("Parent", "x", "y")], T("Offspring", "y", "x")),
]


def provable(kb, goal):
    return any(True for _ in fol_bc_ask(kb, goal))


check("3: Tam is a bird", provable(bird_kb, T("Bird", "Tam")), True)
check("3(e): Doug is Tam's offspring", provable(bird_kb, T("Offspring", "Doug", "Tam")), True)
check("3(b): so Doug is a bird too", provable(bird_kb, T("Bird", "Doug")), True)
check("3: nothing proves Nessie is a bird", provable(bird_kb, T("Bird", "Nessie")), False)
report()

## Where this goes next

`tt_entails` is exponential in the number of symbols, and `pl_resolution` is no better in the worst case, which is what question 1 of Logic 1 means by co-NP-complete.
Real systems restrict the language instead: Horn clauses give the forward and backward chaining of question 5, which run in linear time, and that restriction is what Prolog and the planners of Lectures 7 and 8 are built on.

One remark for the session.
The occurs check costs a line and is omitted from most Prolog implementations, because it turns unification from almost constant time into linear time in the size of the terms.
The price is an engine that is unsound, in the precise sense question 4(b) demonstrates.
